# Swaption Training Data Generation
## 3-Factor Exponential Cheyette Model — Differential Machine Learning Dataset

---

## Conceptual Introduction

### From caplet to swaption: what changes?

The caplet training data (notebook 07) had a clean structure: one maturity pair
$(T_C, T_B)$, one strike $K$, one rate level $f_0$. The swaption is structurally
richer in three ways.

**1. A two-dimensional maturity space.**
A swaption is defined by two independent dates: the option expiry $T_0$ and the
underlying swap maturity $T_N = T_0 + \tau$. These are genuinely independent —
the market quotes a full volatility matrix with expiry on one axis and swap tenor
$\tau$ on the other. Standard market dimensions are:

$$T_0 \in \{1\text{M}, 3\text{M}, 6\text{M}, 1\text{Y}, 2\text{Y}, 3\text{Y},
             5\text{Y}, 7\text{Y}, 10\text{Y}\}
\quad \times \quad
\tau \in \{1\text{Y}, 2\text{Y}, 3\text{Y}, 5\text{Y}, 7\text{Y}, 10\text{Y}\}$$

**2. A richer discount structure.**
The caplet discounts with a single bond $B(0, T_B)$. The swaption discounts with
the **annuity factor**

$$A(0) = \delta \sum_{i=1}^{n} B(0, T_i) = \delta \sum_{i=1}^{n} e^{-f_0 T_i}$$

which depends on the full payment schedule $(T_0, \tau, \delta)$. The annuity is both
the numéraire of the annuity measure and the normalising factor for the normalised
swaption price $V/A(0)$.

**3. The accrual period $\delta$ is a genuine input.**
In the caplet, $\delta = T_B - T_C$ is implied by $(T_C, T_B)$. In the swaption,
the fixed leg accrual period $\delta \in \{0.25, 0.5, 1.0\}$ is a separate market
convention that affects the payment schedule and therefore the annuity and the
par swap rate. A network that ignores $\delta$ cannot generalise across market
conventions.

---

### The "spot" for a swaption

The par swap rate at $t=0$ is

$$S(0) = \frac{B(0, T_0) - B(0, T_N)}{A(0)} = \frac{e^{-f_0 T_0} - e^{-f_0 T_N}}{A(0)}$$

This is the exact analogue of the forward bond price $F$ in the caplet. The
moneyness of the swaption is $m = K/S(0)$: the ratio of the struck fixed rate to
the prevailing par rate.

The rate level $f_0$ enters through two channels, exactly as in the caplet:
- **Moneyness channel**: $S(0)$ depends on $f_0$ via the bond prices, determining
  whether the swaption is ITM, ATM or OTM.
- **Annuity channel**: $A(0)$ scales the overall price level — the swaption price
  is $V = A(0) \cdot g(m, \sigma_{\text{sw}}\sqrt{T_0})$ for some function $g$.

The financially natural feature set is therefore:

| Feature | Expression | Analogy to caplet |
|---------|-----------|-------------------|
| $\sigma_{\text{sw}}\sqrt{T_0}$ | Cheyette swaption vol (from MC) | $v = \sqrt{v^2}$ |
| $S(0)$ | Par swap rate | Forward bond price $F$ |
| $m = K/S(0)$ | Swaption moneyness | Bond moneyness $K^*/F$ |
| $\log m$ | Log-moneyness | $\log(K^*/F)$ |
| $A(0)$ | Annuity factor | $B(0, T_B)$ |
| $V/A(0)$ | Normalised swaption price | $C/B(0, T_B)$ |

---

### Pricing method: Monte Carlo

We use **exact Monte Carlo** under the risk-neutral measure $\mathbb{Q}$. At option
expiry $T_0$, the payer swaption payoff is

$$\text{payoff} = \max\!\bigl(1 - B(T_0, T_N) - K \cdot A(T_0),\; 0\bigr)$$

The Cheyette bond pricing formula gives $B(T_0, T_i)$ as an explicit
exponential-affine function of the state variables $(X_1, X_2, X_3, X_4)$ at $T_0$.
Since the state variables are jointly Gaussian under any forward measure (Theorem 6.1),
we draw exact realisations in a single step — no Euler discretisation.

The swaption price is

$$V(0) = B(0, T_0) \cdot \mathbb{E}^{\mathbb{Q}}\!\bigl[\text{payoff}(X(T_0))\bigr]$$

where the discounting by $B(0, T_0) = e^{-f_0 T_0}$ brings the payoff back to $t=0$.

**Key implementation:** bond prices are computed for all paths and all payment dates
simultaneously via numpy broadcasting, avoiding any Python loop over paths. This
reduces the time per grid point from $\sim 0.3$s (path loop) to $\sim 0.04$s
(vectorised), making the full dataset feasible to generate on a laptop.

**Optional PDE pricing** is also included as a function call wrapping the existing
`price_swaption_pde()` from the repo. It is slower but provides an independent
cross-check for selected grid points.

---

### Greeks via Common Random Numbers (CRN)

Swaption Greeks are computed by **bump-and-reprice with Common Random Numbers**.
CRN means all bumped prices use the identical random draws as the base price —
only the parameters change. Because the noise is shared, it cancels in the finite
difference, giving accurate gradient estimates with far fewer paths than naive
independent simulation.

Concretely: for each grid point, we draw $N$ standard normals once, fix them, and
evaluate the payoff function at five parameter settings:
$(f_0, f_0 \pm \varepsilon_f, K \pm \varepsilon_K, T_0 \pm \varepsilon_t)$.

The three Greeks are:

$$\frac{\partial V}{\partial f_0} \approx \frac{V(f_0+\varepsilon) - V(f_0-\varepsilon)}{2\varepsilon},
\quad
\frac{\partial V}{\partial K} \approx \frac{V(K+\varepsilon) - V(K-\varepsilon)}{2\varepsilon},
\quad
\frac{\partial V}{\partial T_0} \approx \frac{V(T_0+\varepsilon) - V(T_0-\varepsilon)}{2\varepsilon}$$

where $\partial V / \partial T_0$ is the **theta** with $T_N$ held fixed — the
swaption loses time value as expiry approaches, while the underlying swap remains
unchanged.

Bump sizes: $\varepsilon_f = 1\,\text{bp}$, $\varepsilon_K = 1\,\text{bp}$,
$\varepsilon_{T_0} = 1/365$ (one calendar day).

**Why not analytical Greeks?** Unlike the caplet, there is no closed-form Black
formula for the swaption in the Cheyette model without the frozen-annuity
approximation. CRN finite differences with $N = 20{,}000$ paths give Greek
estimates accurate to $\sim 1\%$ — sufficient for DML training.

---

### Grid design

Moneyness spans a wider range than the caplet ($m \in [0.5, 2.0]$ vs $[0.7, 1.4]$)
because swaptions trade further from ATM for tail hedging and structured product
purposes. The grid is non-uniform: finer near ATM where the price surface has the
highest curvature.

The data split follows the caplet convention: split by $f_0$ to test generalisation
across rate environments, holding out the two extreme values for the test set.


## 1. Setup and Core Pricer

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(0)

from cheyette3f.model.parameters import CALIBRATED_PARAMS as p
from cheyette3f.analytics.bonds import G1_1, G2_1, G1_2, G1_3, H
from cheyette3f.monte_carlo.simulation import sample_state_variables, sample_state_variables_qmc

# Optional PDE pricer (slower, for cross-validation)
try:
    from cheyette3f.pde.pde_pricer import price_swaption_pde
    PDE_AVAILABLE = True
    print("PDE pricer available (for optional cross-validation)")
except Exception as e:
    PDE_AVAILABLE = False
    print(f"PDE pricer not loaded: {e}")

print(f"\nModel parameters (fixed throughout):")
print(p.summary())


## 2. Vectorised Bond Price Engine

In [ ]:
def bond_price_batch(t, maturities, X, p, f0):
    """
    Compute B(t, T_i; X^j) for all paths j and maturities T_i simultaneously.

    Uses numpy broadcasting to avoid any Python loop over paths.
    This is the key performance optimisation: O(n_paths * n_maturities) work
    done entirely in numpy, not Python.

    Parameters
    ----------
    t          : current time (scalar)
    maturities : array of shape (n_mat,)
    X          : state variable matrix, shape (n_paths, 4)
    p          : ModelParams
    f0         : flat initial forward rate

    Returns
    -------
    B : array of shape (n_paths, n_mat)
        B[j, i] = B(t, maturities[i]; X[j])
    """
    mats = np.asarray(maturities)

    # G functions: shape (n_mat,) -- purely maturity/time dependent
    g1 = np.array([G1_1(t, T, p) for T in mats])   # = T - t
    g2 = np.array([G2_1(t, T, p) for T in mats])
    g3 = np.array([G1_2(t, T, p) for T in mats])
    g4 = np.array([G1_3(t, T, p) for T in mats])
    h  = np.array([H(t, T, p)    for T in mats])

    # Initial curve ratio B(0,T)/B(0,t): shape (n_mat,)
    B0T   = np.exp(-f0 * mats)
    B0t   = np.exp(-f0 * t) if t > 1e-12 else 1.0
    ratio = B0T / B0t

    # Exponent: shape (n_paths, n_mat) via broadcasting
    # X[:,k] has shape (n_paths,1) broadcast against g_k of shape (1,n_mat)
    exponent = (
        - X[:, 0:1] * g1[np.newaxis, :]
        - X[:, 1:2] * g2[np.newaxis, :]
        - X[:, 2:3] * g3[np.newaxis, :]
        - X[:, 3:4] * g4[np.newaxis, :]
        - h[np.newaxis, :]
    )

    return ratio[np.newaxis, :] * np.exp(exponent)   # (n_paths, n_mat)


# Sanity check: at t=0, X=0, B(0,T) should equal exp(-f0*T)
X_zero = np.zeros((1, 4))
mats   = np.array([1.0, 2.0, 5.0, 10.0])
B_check = bond_price_batch(0.0, mats, X_zero, p, f0=0.05)
B_exact = np.exp(-0.05 * mats)

print("Bond price batch: sanity check at t=0, X=0")
for T, bc, be in zip(mats, B_check[0], B_exact):
    print(f"  T={T:.0f}: batch={bc:.8f}  exact={be:.8f}  err={abs(bc-be):.2e}")


## 3. Derived Quantities: Par Swap Rate, Annuity, Moneyness

In [ ]:
def swap_derived(T0, tau, delta, f0):
    """
    Compute par swap rate S(0), annuity A(0), and payment schedule at t=0.

    Parameters
    ----------
    T0    : option expiry
    tau   : swap tenor (TN - T0)
    delta : accrual period (0.25, 0.5, or 1.0)
    f0    : flat initial forward rate

    Returns
    -------
    dict with: S0, A0, TN, n_periods, act_delta, pay_dates, B0_T0
    """
    TN = T0 + tau
    n_periods = max(1, int(round(tau / delta)))
    act_delta = tau / n_periods   # actual delta (may differ slightly from nominal)
    pay_dates = np.array([T0 + act_delta*(i+1) for i in range(n_periods)])

    B0_T0  = np.exp(-f0 * T0)
    B0_TN  = np.exp(-f0 * TN)
    B0_pays = np.exp(-f0 * pay_dates)
    A0     = act_delta * B0_pays.sum()
    S0     = (B0_T0 - B0_TN) / A0

    return dict(S0=S0, A0=A0, TN=TN, n_periods=n_periods,
                act_delta=act_delta, pay_dates=pay_dates, B0_T0=B0_T0)


# Illustrate for several swaptions
print(f"{'T0':>5} {'tau':>5} {'delta':>6} {'S(0)':>8} {'A(0)':>8} {'TN':>6} {'n_per':>6}")
print("-" * 50)
for T0, tau, delta in [(1,5,0.5),(2,10,0.5),(5,5,1.0),(1,1,0.25),(3,7,0.5)]:
    d = swap_derived(T0, tau, delta, f0=0.05)
    print(f"{T0:>5} {tau:>5} {delta:>6.2f} {d['S0']:>8.4%} {d['A0']:>8.4f} "
          f"{d['TN']:>6.0f} {d['n_periods']:>6}")


## 4. Core Pricer: MC with CRN Greeks

In [ ]:
def price_swaption_with_greeks(T0, tau, K, delta, p, f0, n_paths=20_000,
                                seed=42, receiver=False,
                                eps_f=1e-4, eps_k=1e-4, eps_t=1/365):
    """
    Price a payer (or receiver) swaption and compute 3 Greeks via CRN FD.

    All five payoff evaluations (base + 4 bumps) use identical random draws.
    The noise is common across bumps, so it cancels in the finite difference.
    This gives accurate Greeks with the same n_paths as the price.

    Parameters
    ----------
    T0, tau, K, delta : swaption specification
    p                 : ModelParams (fixed = CALIBRATED_PARAMS)
    f0                : initial flat forward rate
    n_paths           : number of MC paths
    seed              : random seed (for reproducibility)
    receiver          : True = receiver swaption, False = payer
    eps_f, eps_k, eps_t : bump sizes for f0, K, T0

    Returns
    -------
    dict with: price, se, delta_f0, strike_delta, theta, and derived features
    """
    d = swap_derived(T0, tau, delta, f0)
    TN, n_periods, act_delta = d['TN'], d['n_periods'], d['act_delta']

    # Draw state variables ONCE — reused for all bumps (CRN)
    rng = np.random.default_rng(seed)
    X = sample_state_variables(T0, TN, p, n_paths, rng=rng)

    def payoff_vec(T0_, tau_, K_, f0_):
        """Vectorised payoff for all n_paths, given bumped parameters."""
        TN_      = T0_ + tau_
        n_per_   = max(1, int(round(tau_ / delta)))
        act_d_   = tau_ / n_per_
        dates_   = np.array([T0_ + act_d_*(i+1) for i in range(n_per_)])

        # Bond prices: (n_paths, n_periods)
        B_mat    = bond_price_batch(T0_, dates_, X, p, f0_)
        ann_     = act_d_ * B_mat.sum(axis=1)            # (n_paths,)
        B_TN_    = B_mat[:, -1]                           # (n_paths,)
        sv_      = 1.0 - B_TN_ - K_ * ann_               # payer swap value

        pay_     = (np.maximum(sv_, 0.) if not receiver
                    else np.maximum(-sv_, 0.))
        B0_T0_   = np.exp(-f0_ * T0_)                    # discount to t=0
        return B0_T0_ * pay_                              # (n_paths,)

    # ── Base price ──────────────────────────────────────────────────────────
    d_base       = payoff_vec(T0, tau, K, f0)
    price        = d_base.mean()
    se           = d_base.std() / np.sqrt(n_paths)

    # ── dV/df0  (rate delta) ────────────────────────────────────────────────
    delta_f0     = (payoff_vec(T0, tau, K, f0+eps_f) -
                    payoff_vec(T0, tau, K, f0-eps_f)).mean() / (2*eps_f)

    # ── dV/dK  (strike delta) ───────────────────────────────────────────────
    strike_delta = (payoff_vec(T0, tau, K+eps_k, f0) -
                    payoff_vec(T0, tau, K-eps_k, f0)).mean() / (2*eps_k)

    # ── dV/dT0  (theta: TN fixed, T0 moves -> tau changes) ─────────────────
    theta        = (payoff_vec(T0+eps_t, tau-eps_t, K, f0) -
                    payoff_vec(T0-eps_t, tau+eps_t, K, f0)).mean() / (2*eps_t)

    # ── Derived features ─────────────────────────────────────────────────────
    S0           = d['S0']
    A0           = d['A0']
    moneyness    = K / S0 if S0 > 1e-8 else np.nan
    price_norm   = price / A0 if A0 > 1e-8 else np.nan

    # Implied Black swaption vol (from normalised price)
    # Approximate via Newton on Black formula: V/A = S0*N(d1) - K*N(d2)
    iv_sw = _implied_black_vol_swaption(price, S0, K, T0, A0)

    return dict(
        # ── Raw inputs ───────────────────────────────────────────────────────
        T0=T0, tau=tau, TN=TN, K=K, delta=delta, f0=f0,
        # ── Derived features ─────────────────────────────────────────────────
        S0=S0, A0=A0,
        moneyness=moneyness,
        log_moneyness=np.log(moneyness) if (moneyness>0 and np.isfinite(moneyness)) else np.nan,
        B0_T0=d['B0_T0'],
        # ── Labels ───────────────────────────────────────────────────────────
        price=price,
        price_norm=price_norm,
        iv_sw=iv_sw,
        se=se,
        delta_f0=delta_f0,
        strike_delta=strike_delta,
        theta=theta,
    )


def _implied_black_vol_swaption(price, S0, K, T0, A0, tol=1e-9, max_iter=80):
    """Newton-Raphson inversion of Black swaption formula for implied vol."""
    from scipy import stats
    if price <= 0 or S0 <= 0 or K <= 0 or T0 <= 0 or A0 <= 0:
        return np.nan
    # Normalised price: p_norm = price/A0 = S0*N(d1) - K*N(d2)
    p_norm = price / A0
    intrinsic = max(S0 - K, 0.0)
    if p_norm <= intrinsic + 1e-15:
        return 0.0
    # Bisection warmup
    lo, hi = 1e-6, 10.0
    for _ in range(60):
        mid = 0.5*(lo+hi)
        v = mid * np.sqrt(T0)
        if v < 1e-14:
            lo = mid; continue
        d1 = (np.log(S0/K) + 0.5*v**2)/v
        d2 = d1 - v
        p_try = S0*stats.norm.cdf(d1) - K*stats.norm.cdf(d2)
        if p_try < p_norm: lo = mid
        else: hi = mid
    vol = 0.5*(lo+hi)
    # Newton refinement
    for _ in range(max_iter):
        v = vol * np.sqrt(T0)
        if v < 1e-14: break
        d1 = (np.log(S0/K) + 0.5*v**2)/v
        d2 = d1 - v
        p_try = S0*stats.norm.cdf(d1) - K*stats.norm.cdf(d2)
        vega  = S0 * stats.norm.pdf(d1) * np.sqrt(T0)
        if vega < 1e-15: break
        vol  -= (p_try - p_norm) / vega
        vol   = max(vol, 1e-8)
        if abs(p_try - p_norm) < tol: break
    return vol


# Test on a 2Y x 5Y ATM payer swaption
result = price_swaption_with_greeks(2.0, 5.0, None, 0.5, p, 0.05, n_paths=20_000, seed=42)
# K=None means ATM: compute S0 first
d0 = swap_derived(2.0, 5.0, 0.5, 0.05)
result = price_swaption_with_greeks(2.0, 5.0, d0['S0'], 0.5, p, 0.05,
                                     n_paths=20_000, seed=42)

print("2Y x 5Y ATM payer swaption:")
print(f"  S(0) = {result['S0']:.4%}  (par swap rate)")
print(f"  A(0) = {result['A0']:.4f}  (annuity)")
print(f"  K    = {result['K']:.4%}  (= S(0), ATM)")
print()
print(f"  Price      = {result['price']:.6f}  ±{result['se']:.6f}")
print(f"  Price/A(0) = {result['price_norm']:.6f}")
print(f"  Impl. vol  = {result['iv_sw']:.4%}")
print()
print(f"  dV/df0       = {result['delta_f0']:.4f}")
print(f"  dV/dK        = {result['strike_delta']:.4f}  (should be <0)")
print(f"  dV/dT0       = {result['theta']:.6f}  (theta, per year)")
print()
print(f"  Sanity: strike_delta < 0: {'OK' if result['strike_delta'] < 0 else 'FAIL'}")
print(f"  Sanity: price > 0:        {'OK' if result['price'] > 0 else 'FAIL'}")


## 5. Optional PDE Cross-Validation

In [ ]:
# The PDE pricer from the repo provides an independent cross-check.
# It is slower but does not rely on the Gaussian simulation approximation.

if PDE_AVAILABLE:
    print("Cross-validating 2Y x 5Y ATM swaption: MC vs PDE")
    T0_, tau_, f0_ = 2.0, 5.0, 0.05
    d_ = swap_derived(T0_, tau_, 0.5, f0_)
    K_ = d_['S0']   # ATM

    # MC price (fast)
    mc_price = result['price']
    mc_se    = result['se']

    # PDE price (l=2, ls=1 for speed; l=3,ls=2 for production accuracy)
    pde_price = price_swaption_pde(
        T0=T0_, TN=d_['TN'], K=K_, p=p, l=2, ls=1,
        n_time_steps=50, f0=f0_, n_periods=d_['n_periods']
    )
    print(f"  MC:  {mc_price:.5f} ± {mc_se:.5f}")
    print(f"  PDE: {pde_price:.5f}  (l=2, ls=1)")
    print(f"  Diff: {abs(mc_price-pde_price):.5f}  ({abs(mc_price-pde_price)/mc_price:.1%})")
else:
    print("PDE pricer not available — run with full PDE module for cross-validation.")
    print("To use: from cheyette3f.pde.pde_pricer import price_swaption_pde")
    print("        pde_price = price_swaption_pde(T0=2., TN=7., K=K_atm, p=p, l=3, ls=2)")


## 6. Training Grid Design

In [ ]:
# ── Grid definitions ──────────────────────────────────────────────────────────

# Option expiry: quarterly from 0.25Y, then semi-annual, then annual
# Finer at short expiries (vol surface changes fastest there)
T0_GRID = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0])

# Swap tenor: standard market swaption matrix columns
TAU_GRID = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 10.0])

# Accrual period: all three standard market conventions
DELTA_GRID = np.array([0.25, 0.5, 1.0])

# Moneyness m = K/S(0): wider range than caplets, finer near ATM
# Swaptions trade to m=0.5 (deep ITM) and m=2.0 (deep OTM) for tail hedging
MONEYNESS_GRID = np.unique(np.round(np.concatenate([
    np.linspace(0.50, 0.75,  4),    # deep ITM
    np.linspace(0.75, 0.90,  4),    # mild ITM
    np.linspace(0.90, 0.96,  4),    # near ITM (finer)
    np.linspace(0.96, 1.04,  9),    # ATM region (finest)
    np.linspace(1.04, 1.10,  4),    # near OTM (finer)
    np.linspace(1.10, 1.25,  4),    # mild OTM
    np.linspace(1.25, 2.00,  4),    # deep OTM
]), 3))

# Initial forward rate: ZIRP through hiking cycle
F0_GRID = np.unique(np.round(np.concatenate([
    np.linspace(0.010, 0.025, 4),   # ZIRP (fine)
    np.linspace(0.025, 0.060, 5),   # normal range
    np.linspace(0.060, 0.100, 4),   # high rate env
]), 4))

print("Grid dimensions:")
print(f"  T0:        {len(T0_GRID):>3}  {T0_GRID.tolist()}")
print(f"  tau:       {len(TAU_GRID):>3}  {TAU_GRID.tolist()}")
print(f"  delta:     {len(DELTA_GRID):>3}  {DELTA_GRID.tolist()}")
print(f"  moneyness: {len(MONEYNESS_GRID):>3}  {MONEYNESS_GRID.tolist()}")
print(f"  f0:        {len(F0_GRID):>3}  {F0_GRID.tolist()}")
print()

# Count feasible points
raw_count = 0
feasible_count = 0
filter_reasons = {'K_neg': 0, 'n_per_zero': 0, 'S0_tiny': 0}

for T0 in T0_GRID:
    for tau in TAU_GRID:
        for delta in DELTA_GRID:
            n_per = int(round(tau / delta))
            if n_per < 1:
                filter_reasons['n_per_zero'] += len(MONEYNESS_GRID)*len(F0_GRID)
                raw_count += len(MONEYNESS_GRID)*len(F0_GRID)
                continue
            for f0 in F0_GRID:
                d_ = swap_derived(T0, tau, delta, f0)
                S0 = d_['S0']
                for m in MONEYNESS_GRID:
                    raw_count += 1
                    K = m * S0
                    if K <= 0.001:
                        filter_reasons['K_neg'] += 1
                    elif S0 < 1e-5:
                        filter_reasons['S0_tiny'] += 1
                    else:
                        feasible_count += 1

print(f"Raw combinations:   {raw_count:>8,}")
for reason, count in filter_reasons.items():
    print(f"  Filtered ({reason}): {count:>6,}")
print(f"Feasible:           {feasible_count:>8,}")
print()
print(f"At 20k paths/point and ~0.04s/point:")
print(f"  Estimated time: {feasible_count*0.04/60:.0f} minutes")


## 7. Data Generation

In [ ]:
def generate_swaption_dataset(T0_grid, tau_grid, delta_grid, moneyness_grid,
                               f0_grid, p, n_paths=20_000, verbose=True):
    """
    Generate the full swaption training dataset.

    For each (T0, tau, delta, moneyness, f0):
      1. Compute par swap rate S(0) and annuity A(0)
      2. Back out strike K = m * S(0)
      3. Simulate n_paths exact Gaussian draws (CRN shared across bumps)
      4. Compute price + 3 Greeks via vectorised CRN finite differences
      5. Compute derived features (normalised price, implied vol, log-moneyness)

    Returns pd.DataFrame.
    """
    records  = []
    skipped  = 0
    seed_ctr = 0   # unique seed per grid point for reproducibility

    total = sum(
        1 for T0 in T0_grid for tau in tau_grid for delta in delta_grid
        for f0 in f0_grid for m in moneyness_grid
        if int(round(tau/delta)) >= 1
    )

    done = 0
    t_start = time.time()

    for T0 in T0_grid:
        for tau in tau_grid:
            for delta in delta_grid:
                n_per = int(round(tau / delta))
                if n_per < 1:
                    skipped += len(moneyness_grid) * len(f0_grid)
                    continue

                for f0 in f0_grid:
                    d_sw = swap_derived(T0, tau, delta, f0)
                    S0   = d_sw['S0']

                    if S0 < 1e-5:
                        skipped += len(moneyness_grid)
                        continue

                    for m in moneyness_grid:
                        done      += 1
                        seed_ctr  += 1
                        K = m * S0
                        if K <= 0.001:
                            skipped += 1
                            continue

                        try:
                            result = price_swaption_with_greeks(
                                T0=T0, tau=tau, K=K, delta=delta,
                                p=p, f0=f0, n_paths=n_paths,
                                seed=seed_ctr
                            )
                        except Exception:
                            skipped += 1
                            continue

                        if (not np.isfinite(result['price'])
                                or result['price'] < 0):
                            skipped += 1
                            continue

                        records.append(result)

                        if verbose and done % 200 == 0:
                            elapsed = time.time() - t_start
                            rate    = done / elapsed
                            eta     = (total - done) / rate / 60
                            print(f"  {done:>6,}/{total:,} "
                                  f"({100*done/total:.1f}%)  "
                                  f"records={len(records):,}  "
                                  f"ETA={eta:.1f}min", end='\r')

    elapsed = time.time() - t_start
    print(f"\nDone in {elapsed:.0f}s  |  valid={len(records):,}  "
          f"skipped={skipped:,}")
    return pd.DataFrame(records)


print("Starting data generation...")
print("(Using n_paths=20,000 per grid point for ~0.9% relative SE on prices)")
print()

df = generate_swaption_dataset(
    T0_GRID, TAU_GRID, DELTA_GRID, MONEYNESS_GRID, F0_GRID, p,
    n_paths=20_000, verbose=True
)

print(f"\nDataset shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")


## 8. Sanity Checks on Labels

In [ ]:
print("=" * 60)
print("Label sanity checks")
print("=" * 60)

checks = {
    'Price >= 0':          (df['price'] >= -1e-10).all(),
    'Strike delta <= 0':   (df['strike_delta'] <= 1e-10).all(),
    'Price finite':        df['price'].apply(np.isfinite).all(),
    'delta_f0 finite':     df['delta_f0'].apply(np.isfinite).all(),
    'strike_delta finite': df['strike_delta'].apply(np.isfinite).all(),
    'theta finite':        df['theta'].apply(np.isfinite).all(),
    'S0 > 0':              (df['S0'] > 0).all(),
    'A0 > 0':              (df['A0'] > 0).all(),
    'moneyness > 0':       (df['moneyness'] > 0).all(),
}

all_pass = True
for name, ok in checks.items():
    print(f"  {name:<30} {'PASS' if ok else 'FAIL'}")
    if not ok: all_pass = False

print()
print("Label statistics:")
label_cols = ['price', 'price_norm', 'delta_f0', 'strike_delta', 'theta']
print(df[label_cols].describe().T[['min','mean','50%','max','std']].round(5))

print()
print("DML normalisation constants (std dev per label):")
for col in label_cols:
    std = df[col].std()
    print(f"  {col:<20} std={std:.5f}")


## 9. Greek Verification — CRN vs Independent Simulation

In [ ]:
# Verify CRN Greeks against independent (high-N) simulation
# Use a separate large simulation as ground truth for selected points

print("Verifying CRN Greeks against high-N independent simulation...")
print(f"{'Point':<30} {'Greek':<14} {'CRN':>10} {'Indep(100k)':>12} {'rel_err':>10}")
print("-" * 75)

test_points = [
    (1.0, 5.0, 0.5, 0.05),    # 1Y x 5Y, semi-annual, 5%
    (2.0, 10.0, 0.5, 0.03),   # 2Y x 10Y, semi-annual, 3% (ZIRP-ish)
    (5.0, 5.0, 1.0, 0.07),    # 5Y x 5Y, annual, 7% (high rate)
]

for T0_, tau_, delta_, f0_ in test_points:
    d0_ = swap_derived(T0_, tau_, delta_, f0_)
    K_  = d0_['S0']   # ATM
    eps = 1e-4

    # CRN estimate (same seed = same draws)
    r_crn = price_swaption_with_greeks(T0_, tau_, K_, delta_, p, f0_,
                                        n_paths=20_000, seed=999)

    # Independent high-N estimate for each Greek
    n_ind = 100_000
    def ind_price(T0__, tau__, K__, f0__):
        rng_ = np.random.default_rng(12345)
        d__  = swap_derived(T0__, tau__, delta_, f0__)
        X_   = sample_state_variables(T0__, d__['TN'], p, n_ind, rng=rng_)
        B_   = bond_price_batch(T0__, d__['pay_dates'], X_, p, f0__)
        ann_ = d__['act_delta'] * B_.sum(axis=1)
        sv_  = 1. - B_[:,-1] - K__ * ann_
        pay_ = np.maximum(sv_, 0.)
        return (np.exp(-f0__ * T0__) * pay_).mean()

    df0_ind = (ind_price(T0_, tau_, K_, f0_+eps) -
               ind_price(T0_, tau_, K_, f0_-eps)) / (2*eps)
    dK_ind  = (ind_price(T0_, tau_, K_+eps, f0_) -
               ind_price(T0_, tau_, K_-eps, f0_)) / (2*eps)

    lbl = f"T0={T0_}Y tau={tau_}Y f0={f0_:.0%}"
    for greek_name, crn_val, ind_val in [
        ('delta_f0', r_crn['delta_f0'], df0_ind),
        ('strike_delta', r_crn['strike_delta'], dK_ind),
    ]:
        rel = abs(crn_val - ind_val) / (abs(ind_val) + 1e-12)
        ok = "OK" if rel < 0.05 else "CHECK"
        print(f"  {lbl:<30} {greek_name:<14} {crn_val:>10.4f} {ind_val:>12.4f} {rel:>10.2%}  {ok}")


## 10. Visualisation of the Swaption Surface

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

delta_fix = 0.5
f0_fix    = 0.05

# ── Panel A: Price surface — expiry × tenor at ATM ──────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
sub_atm = df[(df['delta'] == delta_fix) & (df['f0'].between(f0_fix-0.001, f0_fix+0.001))
             & (df['moneyness'].between(0.995, 1.005))].copy()

if len(sub_atm) > 0:
    pivot = sub_atm.pivot_table(index='T0', columns='tau', values='price_norm', aggfunc='mean')
    im1 = ax1.imshow(pivot.values, aspect='auto', cmap='viridis',
                     origin='lower', extent=[0,len(pivot.columns),0,len(pivot.index)])
    ax1.set_xticks(range(len(pivot.columns)))
    ax1.set_xticklabels([f'{c:.0f}Y' for c in pivot.columns], fontsize=8)
    ax1.set_yticks(range(len(pivot.index)))
    ax1.set_yticklabels([f'{r:.1f}Y' for r in pivot.index], fontsize=8)
    plt.colorbar(im1, ax=ax1, shrink=0.8)
    ax1.set_title('ATM Normalised Price V/A(0)
(delta=0.5, f0=5%)', fontsize=10)
    ax1.set_xlabel('Swap Tenor'); ax1.set_ylabel('Option Expiry')

# ── Panel B: Implied vol surface ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
if len(sub_atm) > 0:
    pivot_iv = sub_atm.pivot_table(index='T0', columns='tau', values='iv_sw', aggfunc='mean')
    im2 = ax2.imshow(pivot_iv.values*100, aspect='auto', cmap='RdYlGn_r',
                     origin='lower', extent=[0,len(pivot_iv.columns),0,len(pivot_iv.index)])
    ax2.set_xticks(range(len(pivot_iv.columns)))
    ax2.set_xticklabels([f'{c:.0f}Y' for c in pivot_iv.columns], fontsize=8)
    ax2.set_yticks(range(len(pivot_iv.index)))
    ax2.set_yticklabels([f'{r:.1f}Y' for r in pivot_iv.index], fontsize=8)
    plt.colorbar(im2, ax=ax2, shrink=0.8, label='%')
    ax2.set_title('ATM Implied Black Swaption Vol (%)
(delta=0.5, f0=5%)', fontsize=10)
    ax2.set_xlabel('Swap Tenor'); ax2.set_ylabel('Option Expiry')

# ── Panel C: Smile — price_norm vs moneyness for several expiries ─────────────
ax3 = fig.add_subplot(gs[0, 2])
sub_smile = df[(df['delta'] == delta_fix) & (df['tau'] == 5.0)
               & (df['f0'].between(f0_fix-0.001, f0_fix+0.001))].copy()
colors_exp = plt.cm.plasma(np.linspace(0.1, 0.9, len(T0_GRID)))
for T0_, col in zip(T0_GRID, colors_exp):
    slc = sub_smile[np.abs(sub_smile['T0']-T0_)<0.01].sort_values('moneyness')
    if len(slc) > 2:
        ax3.plot(slc['moneyness'], slc['price_norm']*100, '-o', color=col,
                 markersize=3, linewidth=1.5, label=f'T0={T0_:.1f}Y')
ax3.axvline(1.0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax3.set_xlabel('Moneyness K/S(0)'); ax3.set_ylabel('Normalised Price × 100')
ax3.set_title('Price Smile — 5Y Tenor Swaption
(delta=0.5, f0=5%)', fontsize=10)
ax3.legend(fontsize=7, ncol=2); ax3.grid(True, alpha=0.3)

# ── Panel D: Rate delta vs moneyness ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
for T0_, col in zip([1., 2., 5., 10.], plt.cm.viridis(np.linspace(0.1,0.9,4))):
    slc = sub_smile[np.abs(sub_smile['T0']-T0_)<0.01].sort_values('moneyness')
    if len(slc) > 2:
        ax4.plot(slc['moneyness'], slc['delta_f0'], '-o', color=col,
                 markersize=3, linewidth=1.5, label=f'T0={T0_:.0f}Y')
ax4.axvline(1.0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax4.set_xlabel('Moneyness'); ax4.set_ylabel('dV/df0')
ax4.set_title('Rate Delta — 5Y Tenor', fontsize=10)
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

# ── Panel E: Strike delta vs moneyness ───────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
for T0_, col in zip([1., 2., 5., 10.], plt.cm.viridis(np.linspace(0.1,0.9,4))):
    slc = sub_smile[np.abs(sub_smile['T0']-T0_)<0.01].sort_values('moneyness')
    if len(slc) > 2:
        ax5.plot(slc['moneyness'], slc['strike_delta'], '-o', color=col,
                 markersize=3, linewidth=1.5, label=f'T0={T0_:.0f}Y')
ax5.axvline(1.0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax5.axhline(0.0, color='black', linewidth=0.7)
ax5.set_xlabel('Moneyness'); ax5.set_ylabel('dV/dK  (< 0)')
ax5.set_title('Strike Delta — 5Y Tenor', fontsize=10)
ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

# ── Panel F: Label distributions ─────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
label_plot = ['price_norm', 'delta_f0', 'strike_delta', 'theta']
for col, color in zip(label_plot, ['steelblue','tomato','goldenrod','mediumpurple']):
    data = df[col].dropna()
    data_std = (data - data.mean()) / data.std()
    ax6.hist(data_std, bins=50, alpha=0.45, color=color,
             edgecolor='none', density=True, label=col)
ax6.set_xlabel('(Label - mean) / std'); ax6.set_ylabel('Density')
ax6.set_title('Standardised Label Distributions', fontsize=10)
ax6.legend(fontsize=8); ax6.grid(True, alpha=0.3)

plt.suptitle('Swaption Training Data — 3-Factor Cheyette  |  '
             f'N={len(df):,} records', fontsize=13, fontweight='bold')
plt.savefig('../notebooks/swaption_surfaces.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Train / Validation / Test Split and Export

In [ ]:
# Split by f0 value (same strategy as caplet dataset)
f0_sorted  = np.sort(F0_GRID)
test_f0s   = {f0_sorted[0], f0_sorted[-1]}        # ZIRP + peak rate
val_f0s    = set(f0_sorted[2::4])
train_f0s  = set(f0_sorted) - test_f0s - val_f0s

df_train = df[df['f0'].isin(train_f0s)].copy()
df_val   = df[df['f0'].isin(val_f0s)].copy()
df_test  = df[df['f0'].isin(test_f0s)].copy()

print("Data split by f0 (testing generalisation across rate environments):")
print(f"  Train f0 values: {sorted(train_f0s)}")
print(f"  Val   f0 values: {sorted(val_f0s)}")
print(f"  Test  f0 values: {sorted(test_f0s)}")
print()
print(f"  Train: {len(df_train):>7,} records")
print(f"  Val:   {len(df_val):>7,} records")
print(f"  Test:  {len(df_test):>7,} records")

OUT_DIR = Path('../data')
OUT_DIR.mkdir(exist_ok=True)

for name, split in [('swaption_full', df), ('swaption_train', df_train),
                     ('swaption_val', df_val), ('swaption_test', df_test)]:
    path = OUT_DIR / f'{name}.csv'
    split.to_csv(path, index=False)
    print(f"  Saved {str(path):<45} {len(split):>7,} rows  "
          f"{path.stat().st_size/1e6:.1f} MB")

print()
print("Column manifest:")
for col in df.columns:
    group = ('INPUT (raw)'      if col in ['T0','tau','TN','K','delta','f0']
             else 'FEATURE'     if col in ['S0','A0','moneyness','log_moneyness','B0_T0']
             else 'LABEL'       if col in ['price','price_norm','iv_sw',
                                           'delta_f0','strike_delta','theta']
             else 'DIAGNOSTIC')
    print(f"  {col:<22} {group}")


## 12. Differential ML — Loss Function Guidance

Following the caplet DML setup, the recommended joint loss is:

$$\mathcal{L} = \frac{w_V}{\sigma_V^2}\|V - \hat{V}\|^2
+ \frac{w_\Delta}{\sigma_\Delta^2}\|\partial_{f_0}V - \widehat{\partial_{f_0}V}\|^2
+ \frac{w_K}{\sigma_K^2}\|\partial_K V - \widehat{\partial_K V}\|^2
+ \frac{w_\theta}{\sigma_\theta^2}\|\partial_{T_0}V - \widehat{\partial_{T_0}V}\|^2$$

where $\sigma_\cdot$ are the empirical standard deviations of each label (printed below).
Train on `price_norm` = $V/A(0)$ rather than raw price to remove the annuity scaling.


In [ ]:
print("DML loss normalisation constants:")
print(f"{'Label':<22} {'std':>10} {'suggested weight'}")
print("-" * 50)
for col, name, w in [
    ('price_norm',   'Normalised price V/A(0)', '1.0'),
    ('delta_f0',     'Rate delta dV/df0',        '2.0'),
    ('strike_delta', 'Strike delta dV/dK',       '2.0'),
    ('theta',        'Theta dV/dT0',             '1.0'),
]:
    std = df[col].std()
    print(f"  {name:<20} {std:>10.5f}  weight={w}")
print()
print("Input features for the network:")
print("  Raw:     T0, tau, delta, K, f0")
print("  Natural: S0, A0, moneyness (=K/S0), log_moneyness, B0_T0")
print()
print("Output targets:")
print("  Primary:    price_norm  (V/A(0) — removes annuity scale)")
print("  Greeks:     delta_f0, strike_delta, theta")
print("  Alt target: iv_sw (Black implied vol — bounded, smooth)")
